
# Parcial Corte 2 — ETL Completo Cafetería

Este notebook cumple:
- ETL completo
- Limpieza real (1NF y 3NF)
- Validación de datos
- Base de datos SQLite
- CRUD completo

In [ ]:

import pandas as pd
import sqlite3


## FASE 1 — Carga de datos

In [ ]:
# Cargar datos sucios
df_prod_dirty = pd.read_csv('parcial_2_productos_sucios.csv')
df_cli_dirty = pd.read_csv('parcial_2_clientes_sucios.csv')
df_prov_dirty = pd.read_csv('parcial_2_proveedores_sucios.csv')

# Verificar estructura REAL (evita errores)
print("PRODUCTOS:", df_prod_dirty.columns)
print("CLIENTES:", df_cli_dirty.columns)
print("PROVEEDORES:", df_prov_dirty.columns)

## FASE 2 — Limpieza de datos 

In [ ]:
# PRODUCTOS

df_prod = df_prod_dirty.copy()

if 'producto_categoria' in df_prod.columns:
    df_prod[['nombre_producto', 'categoria']] = df_prod['producto_categoria'].str.split(' - ', n=1, expand=True)

if 'precio' in df_prod.columns:
    df_prod['precio'] = df_prod['precio'].astype(str).str.replace('$','', regex=False)
    df_prod['precio'] = pd.to_numeric(df_prod['precio'], errors='coerce')

if 'stock' in df_prod.columns:
    df_prod['stock'] = df_prod['stock'].fillna(0)
    df_prod['stock'] = pd.to_numeric(df_prod['stock'], errors='coerce').fillna(0).astype(int)

# Estandarizar texto
for col in df_prod.select_dtypes(include='object').columns:
    df_prod[col] = df_prod[col].str.title()


# CLIENTES

df_cli = df_cli_dirty.copy()

if 'cliente_tipo' in df_cli.columns:
    df_cli[['nombre_cliente', 'tipo_cliente']] = df_cli['cliente_tipo'].str.split(' - ', n=1, expand=True)

if 'edad' in df_cli.columns:
    df_cli = df_cli.drop(columns=['edad'])

df_cli = df_cli.fillna('No Registra')

for col in df_cli.select_dtypes(include='object').columns:
    df_cli[col] = df_cli[col].str.title()


# PROVEEDORES

df_prov = df_prov_dirty.copy()

if 'empresa_ciudad' in df_prov.columns:
    df_prov[['nombre_empresa', 'ciudad']] = df_prov['empresa_ciudad'].str.split(' - ', n=1, expand=True)

df_prov = df_prov.fillna('No Registra')

for col in df_prov.select_dtypes(include='object').columns:
    df_prov[col] = df_prov[col].str.title()

df_prod.head()

## Exportación

In [ ]:

df_prod.to_csv('parcial_2_productos_limpios.csv', index=False)
df_cli.to_csv('parcial_2_clientes_limpios.csv', index=False)
df_prov.to_csv('parcial_2_proveedores_limpios.csv', index=False)


## FASE 3 — Base de Datos

In [ ]:

conn = sqlite3.connect('parcial_2_cafeteria.db')
cursor = conn.cursor()

df_prod.to_sql('productos', conn, if_exists='replace', index=False)
df_cli.to_sql('clientes', conn, if_exists='replace', index=False)
df_prov.to_sql('proveedores', conn, if_exists='replace', index=False)

NameError: name 'sqlite3' is not defined

In [ ]:
print(pd.read_sql("PRAGMA table_info(proveedores);", conn))

## FASE 4 — Tabla ventas

In [ ]:

cursor.execute("""
CREATE TABLE IF NOT EXISTS ventas (
    id_venta INTEGER PRIMARY KEY AUTOINCREMENT,
    id_cliente INTEGER NOT NULL,
    id_producto INTEGER NOT NULL,
    id_proveedor INTEGER NOT NULL,
    cantidad INTEGER NOT NULL,
    total_venta REAL NOT NULL,
    fecha_venta TEXT NOT NULL,
    FOREIGN KEY (id_cliente) REFERENCES clientes(id_cliente),
    FOREIGN KEY (id_producto) REFERENCES productos(id_producto),
    FOREIGN KEY (id_proveedor) REFERENCES proveedores(nit_proveedor)
);
""")

conn.commit()

In [ ]:
print(pd.read_sql("SELECT * FROM clientes LIMIT 5;", conn))
print(pd.read_sql("SELECT * FROM productos LIMIT 5;", conn))
print(pd.read_sql("SELECT * FROM proveedores LIMIT 5;", conn))

## CRUD

In [ ]:

cursor.execute("INSERT INTO ventas (id_cliente,id_producto,id_proveedor,cantidad,total_venta,fecha_venta) VALUES (1,1,1,2,10000,'2026-04-01')")
cursor.execute("INSERT INTO ventas (id_cliente,id_producto,id_proveedor,cantidad,total_venta,fecha_venta) VALUES (2,2,1,1,5000,'2026-04-02')")
cursor.execute("INSERT INTO ventas (id_cliente,id_producto,id_proveedor,cantidad,total_venta,fecha_venta) VALUES (3,3,2,3,15000,'2026-04-03')")
conn.commit()

In [ ]:

# READ
pd.read_sql_query("""
SELECT v.id_venta, c.nombre_cliente, p.nombre_producto, v.total_venta
FROM ventas v
INNER JOIN clientes c ON v.id_cliente = c.id_cliente
INNER JOIN productos p ON v.id_producto = p.id_producto;
""", conn)


In [ ]:

# UPDATE
cursor.execute("UPDATE ventas SET cantidad=5, total_venta=25000 WHERE id_venta=1")
conn.commit()


In [ ]:

# DELETE
cursor.execute("DELETE FROM ventas WHERE id_venta=2")
conn.commit()


In [ ]:
conn.close()